In [2]:
# Importando las bibliotecas necesarias
from urllib.request import urlopen, Request  # Para abrir URLs y manejar solicitudes
from urllib.error import HTTPError  # Para manejar errores HTTP
import time  # Para implementar retrasos si es necesario

In [8]:
# Función para extraer enlaces del contenido HTML
def getLinks(html, max_links=10):
    url = []  # Lista para almacenar las URLs extraídas
    cursor = 0  # Cursor para rastrear la posición en el contenido HTML
    nlinks = 0  # Contador para el número de enlaces extraídos

    # Bucle para extraer enlaces hasta que se alcance el máximo o no se encuentren más enlaces
    while cursor >= 0 and nlinks < max_links:
        start_link = html.find("a href", cursor)  # Encontrar el inicio de un enlace
        if start_link == -1:  # Si no se encuentran más enlaces, devolver la lista de URLs
            return url

        start_quote = html.find('"', start_link)  # Encontrar la comilla de apertura de la URL
        end_quote = html.find('"', start_quote + 1)  # Encontrar la comilla de cierre de la URL
        url.append(html[start_quote + 1: end_quote])  # Extraer y agregar la URL a la lista
        cursor = end_quote + 1  # Mover el cursor más allá de esta URL
        nlinks += 1  # Incrementar el contador de enlaces

    return url  # Devolver la lista de URLs

In [17]:
# Ejemplo de uso:
# Define la URL del sitio web que se va a analizar
url = 'https://python.org/'
from urllib.request import urlopen, Request
import gzip

# Crea una solicitud HTTP con una cabecera de 'User-Agent'
req = Request(url, headers={'User-Agent': 'Magic Browser'})

# Abre la URL utilizando la solicitud creada y almacena la respuesta en la variable 'con'
con = urlopen(req)

# Check for content encoding and handle accordingly
if con.info().get('Content-Encoding') == 'gzip':
    # Decompress if the content is Gzip-encoded
    html_content = gzip.decompress(con.read()).decode('utf-8')
else:
    # Attempt to decode with the correct encoding (assume UTF-8 if none specified)
    html_content = con.read().decode(con.headers.get_content_charset() or 'utf-8')

# Print the content length for verification (optional)
print(f"Content Length: {len(html_content)} characters")

# Use the getLinks function
links = getLinks(html_content)

# Print the extracted links
print(links)

# Lee el contenido de la respuesta y decodifica el contenido de bytes a una cadena


Content Length: 51269 characters
['http://browsehappy.com/', '#content', '/', 'https://www.python.org/psf/', 'https://docs.python.org', 'https://pypi.org/', '/jobs/', '/community/', '/', 'https://psfmember.org/civicrm/contribute/transact?reset=1&id=2']


In [18]:


# Define la clase Spider para el rastreo web
class Spider:
    # Inicializador o constructor para la clase Spider
    def __init__(self, starting_url, crawl_domain, max_iter):
        self.crawl_domain = crawl_domain  # El dominio dentro del cual la araña rastreará
        self.max_iter = max_iter  # El número máximo de páginas a rastrear
        self.links_to_crawl = [starting_url]  # Cola de enlaces a rastrear
        self.links_visited = []  # Lista para hacer un seguimiento de los enlaces visitados
        self.collection = []  # Lista para almacenar los datos recolectados

    # Método para recuperar el contenido HTML desde una URL
    def retrieveHtml(self):
        try:
            # Abrir la URL y leer la respuesta
            socket = urlopen(self.url)
            # Decodificar la respuesta usando la codificación 'latin-1'
            self.html = socket.read().decode('latin-1')
            return 0  # Retornar 0 si es exitoso
        except HTTPError as e:
            # Si ocurre un error HTTP, imprimir el error y retornar -1
            print(f"Se ha encontrado un error HTTP: {e}")
            return -1

    # Método principal para controlar el proceso de rastreo
    def run(self):
        # Continuar el rastreo mientras haya enlaces por rastrear y no se haya alcanzado max_iter
        while self.links_to_crawl and len(self.collection) < self.max_iter:
            # Obtener el siguiente enlace a rastrear
            self.url = self.links_to_crawl.pop(0)
            print(f"Actualmente rastreando: {self.url}")
            # Agregar el enlace a la lista de enlaces visitados
            self.links_visited.append(self.url)
            # Si la recuperación del HTML es exitosa, almacenar el HTML y encontrar nuevos enlaces
            if self.retrieveHtml() >= 0:
                self.storeHtml()
                self.retrieveAndValidateLinks()

    # Método para recuperar y validar los enlaces en el contenido HTML
    def retrieveAndValidateLinks(self):
        # Obtener una lista de enlaces desde el contenido HTML actual
        items = getLinks(self.html)
        # Lista temporal para almacenar los enlaces válidos
        tmpList = []

        # Iterar sobre los enlaces encontrados
        for item in items:
            item = item.strip('"')  # Eliminar comillas extra

            # Verificar si el enlace es una URL absoluta que contiene el dominio a rastrear
            if self.crawl_domain in item and item.startswith('http'):
                tmpList.append(item)
            # Manejar enlaces relativos
            elif item.startswith('/'):
                # Construir la URL completa usando el dominio a rastrear y el enlace relativo
                tmpList.append('https://' + self.crawl_domain + item)
            # Manejar posibles enlaces relativos sin barra inicial (asumiendo que no son URLs absolutas)
            elif not item.startswith('http'):
                # Construir la URL completa asumiendo que es un enlace relativo
                tmpList.append('https://' + self.crawl_domain + '/' + item)

        # Agregar los enlaces válidos y no visitados a la cola de rastreo
        for item in tmpList:
            if item not in self.links_visited and item not in self.links_to_crawl:
                self.links_to_crawl.append(item)
                print(f'Agregado a la cola de rastreo: {item}')

    # Método para almacenar el contenido HTML y los metadatos asociados
    def storeHtml(self):
        # Crear un diccionario para representar el documento
        doc = {
            'url': self.url,  # URL de la página
            'date': time.strftime("%d/%m/%Y"),  # Fecha actual
            'html': self.html  # Contenido HTML de la página
        }
        # Agregar el documento a la colección
        self.collection.append(doc)
        print(f"HTML almacenado desde: {self.url}")


In [19]:
# Asumiendo que la clase Spider está definida previamente con la función getLinks implementada


# Crear una instancia de la clase Spider con la URL inicial, el dominio a rastrear y el número máximo de iteraciones

spider = Spider('http://books.toscrape.com/', 'books.toscrape.com', 20)


# Iniciar el proceso de rastreo.

spider.run()


# Después de ejecutar el rastreo, `spider.collection` contendrá el HTML de hasta 20 páginas de 'books.toscrape.com'.

# Cada entrada en la colección incluirá la URL, la fecha en que se rastreó y el contenido HTML de la página.

Actualmente rastreando: http://books.toscrape.com/
HTML almacenado desde: http://books.toscrape.com/
Agregado a la cola de rastreo: https://books.toscrape.com/index.html
Agregado a la cola de rastreo: https://books.toscrape.com/catalogue/category/books_1/index.html
Agregado a la cola de rastreo: https://books.toscrape.com/catalogue/category/books/travel_2/index.html
Agregado a la cola de rastreo: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Agregado a la cola de rastreo: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Agregado a la cola de rastreo: https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html
Agregado a la cola de rastreo: https://books.toscrape.com/catalogue/category/books/classics_6/index.html
Agregado a la cola de rastreo: https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html
Agregado a la cola de rastreo: https://books.toscrape.com/catalogue/category/books/romanc

In [11]:
# Importar la función 'urlopen' del módulo 'urllib.request'.

# La función urlopen permite abrir una URL en Python.

# La utilización de esta función es crucial para iniciar el proceso de scraping, ya que nos permite acceder al contenido de la web.

from urllib.request import urlopen


# Usar la función 'urlopen' para abrir una URL.

# La URL proporcionada puede ser cualquier página web accesible. En este ejemplo, estamos utilizando la página de Mitsubishi Electric.

# El resultado es un objeto que contiene información sobre la página web, como el código HTML, los encabezados HTTP y otros metadatos.

source = urlopen("http://# Leer el contenido de la respuesta HTTP.

# Al llamar a source.read(), obtenemos el contenido crudo en formato binario.

something = source.read()


# Imprimir el contenido crudo de la página.

# Este contenido está en bytes, lo que significa que no será legible como texto en la consola aún.

# Imprimir esto directamente mostrará una secuencia de bytes que representa el código HTML de la página web.

print(something)mitsubishielectric.es/aire-acondicionado")


# Imprimir el objeto de respuesta HTTP.

# Es importante notar que esto no muestra el contenido HTML directamente, sino una representación del objeto de respuesta HTTP.

# Este objeto de respuesta contiene metadatos como el estado de la solicitud, las cabeceras HTTP y más.

print(source)

In [12]:
# Leer el contenido de la respuesta HTTP.

# Al llamar a source.read(), obtenemos el contenido crudo en formato binario.

something = source.read()


# Imprimir el contenido crudo de la página.

# Este contenido está en bytes, lo que significa que no será legible como texto en la consola aún.

# Imprimir esto directamente mostrará una secuencia de bytes que representa el código HTML de la página web.

print(something)

b'<!doctype html>\n<html lang="es" prefix="og: https://ogp.me/ns#">\n\n<head>\n\n<script>window.dataLayer = window.dataLayer || [];</script>\n<!-- Google Tag Manager -->\n<script>(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({\'gtm.start\':\nnew Date().getTime(),event:\'gtm.js\'});var f=d.getElementsByTagName(s)[0],\nj=d.createElement(s),dl=l!=\'dataLayer\'?\'&l=\'+l:\'\';j.async=true;j.src=\n\'https://www.googletagmanager.com/gtm.js?id=\'+i+dl;f.parentNode.insertBefore(j,f);\n})(window,document,\'script\',\'dataLayer\',\'GTM-NJHKGD2\');</script>\n\t<!-- End Google Tag Manager -->\n\n    <!--Start of LiveBeep Script-->\n    <script type="text/javascript">\n        (function(d, s, id) {\n            if (d.getElementById(id)) {\n                return;\n            }\n            var u = \'//www.livebeep.com/\' + d.domain + \'/eye.js?\';\n            if ((h = d.location.href.split(/#ev!/)[1])) u += \'?_e=\' + h;\n            else if ((r = /.*\\_evV=(\\w+)\\b.*/).test(c = d.cookie)) u += \

In [13]:
# Decodificar el contenido binario a una cadena de texto.

# Al usar .decode('utf-8'), transformamos los datos binarios en texto legible en formato UTF-8.

decoded_content = something.decode('utf-8')


# Imprimir el contenido decodificado.

# Ahora el contenido es una cadena de texto, y se puede visualizar como el código HTML de la página web.

print(decoded_content)

<!doctype html>
<html lang="es" prefix="og: https://ogp.me/ns#">

<head>

<script>window.dataLayer = window.dataLayer || [];</script>
<!-- Google Tag Manager -->
<script>(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
})(window,document,'script','dataLayer','GTM-NJHKGD2');</script>
	<!-- End Google Tag Manager -->

    <!--Start of LiveBeep Script-->
    <script type="text/javascript">
        (function(d, s, id) {
            if (d.getElementById(id)) {
                return;
            }
            var u = '//www.livebeep.com/' + d.domain + '/eye.js?';
            if ((h = d.location.href.split(/#ev!/)[1])) u += '?_e=' + h;
            else if ((r = /.*\_evV=(\w+)\b.*/).test(c = d.cookie)) u += '?_v=' + c.replace(r, '$1');
            var js = d.c

In [14]:
# Ejemplo de uso:

# Supongamos que tienes contenido HTML almacenado en una variable `html_content`

html_content = decoded_content

# Llamarías a la función de esta forma:

links = getLinks(html_content)

# Esto devolvería una lista de URLs extraídas del contenido HTML

links


['https://www.mitsubishielectric.es/aire-acondicionado/',
 'https://www.mitsubishielectric.es/aire-acondicionado/',
 'https://www.mitsubishielectric.es/aire-acondicionado/profesionales/',
 'https://www.mitsubishielectric.es/aire-acondicionado/descubre-mitsubishi-electric/',
 '#',
 'https://www.mitsubishielectric.com/en/index.html',
 '#',
 'https://www.mitsubishielectric.es/aire-acondicionado/es-tu-calefaccion/',
 '#',
 'https://www.mitsubishielectric.es/aire-acondicionado/producto/aire-acondicionado/']